In [ ]:
import time, psutil, os
import multiprocessing
import traceback
from memory_profiler import memory_usage


def run_and_measure_once(f, *args, **kw):
    p = psutil.Process(os.getpid())

    cpu0 = p.cpu_times()
    t0 = time.perf_counter()

    mem, out = memory_usage(
        (f, args, kw),
        interval=0.05,
        max_iterations=1,
        retval=True
    )

    t1 = time.perf_counter()
    cpu1 = p.cpu_times()

    return {
        "result": out,
        "wall_time_s": t1 - t0,
        "cpu_time_s": (cpu1.user - cpu0.user) + (cpu1.system - cpu0.system),
        "peak_mem_MiB": max(mem)
    }



def worker_function(queue, func, *args, **kwargs):

    try:
        result = run_and_measure_once(func, *args, **kwargs)
        queue.put(("success", result))
    except Exception:
        queue.put(("error", traceback.format_exc()))


def run_with_timeout(timeout_sec, func, *args, **kwargs):
    queue = multiprocessing.Queue()

    p = multiprocessing.Process(
        target=worker_function,
        args=(queue, func, *args),
        kwargs=kwargs
    )

    p.start()
    p.join(timeout=timeout_sec)

    # timeout
    if p.is_alive():
        p.terminate()
        p.join()

        return {
            "status": "timeout",
            "message": f"No solution found within {timeout_sec} seconds."
        }

    if not queue.empty():
        status, data = queue.get()

        if status == "success":
            return {
                "status": "success",
                "data": data
            }

        else:
            return {
                "status": "error",
                "message": data
            }

    return {
        "status": "error",
        "message": "Unknown error."
    }


def read_P_file(p_path):
    """
    Read instances (polynomials P) from file.
    input: path for file
    output: list of polynomials P
    """
    P_list = []
    with open(p_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            s = line.strip()
            if not s:
                continue

            if not (s.startswith("[") and s.endswith("]")):
                raise ValueError(f"Line {line_no}: invalid format")

            inner = s[1:-1].strip()

            if inner == "":
                P_list.append([])
                continue

            try:
                P_vec = [int(x.strip()) for x in inner.split(",")]
            except ValueError:
                raise ValueError(f"Line {line_no}: invalid format")

            P_list.append(P_vec)

    return P_list


def read_PQ_file(pq_path):
    """
    Read polynomials PQ from file
    input: path for file
    output: list of polynomials PQ
    """
    P_list = []
    with open(pq_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            s = line.strip()
            if not s:
                continue

            if not (s.startswith("[") and s.endswith("]")):
                raise ValueError(f"line {line_no}: invalid format")

            inner = s[1:-1].strip()

            if inner == "":
                P_list.append([])
                continue

            try:
                P_vec = [int(x.strip()) for x in inner.split(",")]
            except ValueError:
                raise ValueError(f"Line {line_no}: invalid format")

            P_list.append(P_vec)

    return P_list

def read_weight_file(path):
    values = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line: 
                values.append(int(line))
    return values

# ==================
# ISD-for-codeword-search (Stern-style) for low-weight codewords
# along with helpers to build a generator matrix from P(x).

#We have

#a helper to build a generator matrix  G, G from a polynomial  P(x)
#P(x) (both a non-cyclic Toeplitz “convolution” version and a cyclic version—pick what matches our experiment),

#utilities for systematic form with column permutations,

#a Stern-style ISD for codeword search: it looks for a low-weight codeword  c=uG, c=uG of (target) weight  w using the classic “collision on a window  B”
#idea (two lists of partial row-sums collide on the window so weight concentrates off  B


from itertools import combinations
from random import sample, randint, shuffle
from collections import defaultdict

# -----------------------------
#  utilities
# -----------------------------
F2 = GF(2)

def vec(v):
    return vector(F2, v)

def hamming_weight_vec(v):
    return sum(int(b) for b in v)

# ---------------------------------------------------------------
# 1) Build a generator matrix G from a polynomial P(x) over GF(2)
# ---------------------------------------------------------------
def generator_matrix_from_polynomial(P, n, *, cyclic=False):
    """
    Construct a generator matrix G over GF(2) from a polynomial P(x).
    """
    R = P.parent()
    x = R.gen()
    coeffs = list(P)  # low -> high
    degP = P.degree()

    if not cyclic:
        # Non-cyclic Toeplitz-convolution style:
        # Row i : place coeffs at columns [i .. i+degP], no wrap.
        print(n)
        print(degP)
        k = n - degP
        print(k)
        
        if k <= 0:
            raise ValueError("For non-cyclic construction, require n > deg(P).")
        rows = []
        for i in range(k):
            row = [0] * n
            for j, c in enumerate(coeffs):
                if i + j < n:
                    row[i + j] = row[i + j] ^^ int(c)
            rows.append(row)
        return matrix(F2, rows)
    else:
        # Cyclic (circulant) code from P(x) modulo x^n - 1:
        # Start with a length-n vector: coefficients of P padded/truncated to length n
        base = [0] * n
        for j, c in enumerate(coeffs):
            base[j % n] = base[j % n] ^^ int(c)
        # Build all n cyclic shifts; we may row-reduce afterwards to get dimension k.
        rows = []
        cur = base[:]
        for _ in range(n):
            rows.append(cur[:])
            # rotate right by 1 (or left; consistent choice)
            cur = cur[-1:] + cur[:-1]
        G_full = matrix(F2, rows)
        # Optional: row-reduce to get a basis
        G = G_full.echelon_form()
        G = G.delete_rows([i for i in range(G.nrows()) if G.row(i).is_zero()])
        return G

# ------------------------------------------------------------------
# 2) Systematic form [I_k | A] with column permutation bookkeeping
# ------------------------------------------------------------------
def to_systematic_with_colperm(G):
    """
    Put G into (row-reduced) systematic form by permuting columns.
    Returns G_sys, colperm, colperm_inv
    so that G_sys = G * P (column permutation),
    codewords in original order are c = (u * G_sys) * P^{-1}.
    """
    n = G.ncols()
    # Track a working copy with explicit column swaps
    G_work = matrix(G)
    colperm = list(range(n))

    # Greedy: try to find pivots by swapping columns
    r = 0
    for c in range(n):
        if r >= G_work.nrows():
            break
        if G_work[r, c] == 0:
            # find a column >= c with a 1 in row r
            found = False
            for c2 in range(c+1, n):
                if G_work[r, c2] == 1:
                    # swap columns c and c2 in G_work and in colperm
                    G_work.swap_columns(c, c2)
                    colperm[c], colperm[c2] = colperm[c2], colperm[c]
                    found = True
                    break
            if not found:
                # Try to swap down a row that has a 1 in column c
                for r2 in range(r+1, G_work.nrows()):
                    if G_work[r2, c] == 1:
                        G_work.swap_rows(r, r2)
                        found = True
                        break
                if not found:
                    continue  # no pivot here
        # Now pivot at (r,c) if it's 1
        if G_work[r, c] == 1:
            # Clear column c except row r
            for r2 in range(G_work.nrows()):
                if r2 != r and G_work[r2, c] == 1:
                    G_work.add_multiple_of_row(r2, r, 1)
            r += 1

    # Finish with row echelon reduction (keeps column order)
    G_sys = G_work.echelon_form()

    # Build inverse permutation
    colperm_inv = [0]*n
    for i, p in enumerate(colperm):
        colperm_inv[p] = i
    return G_sys, colperm, colperm_inv

def apply_colperm_to_vector(c, colperm_inv):
    """
    Given a codeword c in the permuted-column space, return c in the original column order.
    colperm_inv maps permuted index -> original index.
    """
    n = len(c)
    out = [0]*n
    for i in range(n):
        out[colperm_inv[i]] = int(c[i])
    return vec(out)

# --------------------------------------------------------------
# 3) Stern-style ISD for codeword search (target weight = w)
# --------------------------------------------------------------
def stern_codeword_search(G, w, *,
                          B_size=20,    # |B|, a window of columns to cancel by collisions
                          p=2,          # choose subsets of p rows on each side
                          list_limit=5000,    # cap lists to keep memory bounded
                          iters=2000,   # total outer iterations
                          verbose=True):
    """
    Find a low-weight codeword c = u * G of (target) weight ~ w using a Stern-style collision
    on a chosen window B of columns. Heuristic; success is probabilistic.

    
    Returns
    -------
    (success, c, u, stats) where:
      success : bool
      c       : codeword (vec over GF(2)) in original column order (or None)
      u       : information vector that produced c in G_sys domain (or None)
      stats   : dict with counters
    """
    k, n = G.nrows(), G.ncols()
    if k == 0:
        raise ValueError("G has zero rows.")
    if w <= 0 or w > n:
        raise ValueError("Target weight w must be in [1..n].")
    if B_size <= 0 or B_size > n:
        raise ValueError("B_size must be in [1..n].")
    if p <= 0 or p > k//2:
        raise ValueError("p should be <= floor(k/2) and >= 1.")

    stats = dict(iters=0, collisions=0, candidates=0)

    for it in range(1, iters+1):
        stats['iters'] = it

        # Systematic form with column permutation
        G_sys, colperm, colperm_inv = to_systematic_with_colperm(G)
        k_sys, n_sys = G_sys.nrows(), G_sys.ncols()

        # Heuristic: choose B inside the "A-part" if G_sys appears [I|A]
        # Identify pivot columns (where each row has a leading 1)
        pivots = set()
        for r in range(k_sys):
            for c in range(n_sys):
                if G_sys[r, c] == 1:
                    pivots.add(c)
                    break
        nonpiv = [c for c in range(n_sys) if c not in pivots]
        if len(nonpiv) >= B_size:
            B = sample(nonpiv, B_size)
        else:
            B = sample(range(n_sys), B_size)

        B = sorted(B)
        B_mask = [1 if i in B else 0 for i in range(n_sys)]

        # Split rows into R1 and R2
        rows = list(range(k_sys))
        shuffle(rows)
        mid = k_sys // 2
        R1, R2 = rows[:mid], rows[mid:]
        if len(R1) < p or len(R2) < p:
            continue  # try again

        # Build L1: all (or a random subset of) p-combinations from R1
        # Restrict sums to B (window)
        def restricted_sum(rows_subset):
            s = [0]*n_sys
            for r in rows_subset:
                s = [(s[j] ^^ int(G_sys[r, j])) for j in range(n_sys)]
            # project to B
            return tuple(s[j] for j in B), s  # (key, full_sum)

        # Limit the number of combinations for scalability
        combs_R1 = list(combinations(R1, p))
        if len(combs_R1) > list_limit:
            combs_R1 = sample(combs_R1, list_limit)

        L1 = defaultdict(list)
        for subset in combs_R1:
            key, full = restricted_sum(subset)
            L1[key].append((subset, full))
            if len(L1[key]) > 3:
                L1[key] = L1[key][-3:]

        # Scan R2 on the fly and look for collisions
        combs_R2 = list(combinations(R2, p))
        if len(combs_R2) > list_limit:
            combs_R2 = sample(combs_R2, list_limit)

        found = False
        for subset2 in combs_R2:
            key2, full2 = restricted_sum(subset2)
            if key2 in L1:
                stats['collisions'] += 1
                # Try a few matches in this bucket
                for subset1, full1 in L1[key2]:
                    # Candidate u = xor of the chosen rows
                    u = [0]*k_sys
                    for r in subset1:
                        u[r] = u[r] ^^ 1
                    for r in subset2:
                        u[r] = u[r] ^^ 1
                    u = vec(u)

                    # Build codeword in permuted space, then unpermute
                    c_perm = u * G_sys
                    c = apply_colperm_to_vector(c_perm, colperm_inv)
                    wt = hamming_weight_vec(c)

                    stats['candidates'] += 1
                    if wt>=1 and wt <= w:
                        if verbose:
                            print(f"[iter {it}] success: weight={wt} (target<={w}), "
                                  f"|B|={B_size}, p={p}")
                        return True, c, u, wt, stats
                # (continue scanning)
        # loop continues; new iteration picks fresh B and split
        if verbose and it % max(1, iters//10) == 0:
            print(f"[iter {it}] collisions so far: {stats['collisions']}, "
                  f"candidates: {stats['candidates']}")

    if verbose:
        print("No codeword found within iteration budget.")
    return False, None, None, 0, stats

# --------------------------------------------------------------
# 4) glue with the P, Q, PQ (from the LWPM step)
# --------------------------------------------------------------
def run_isd_after_lwpm(P, n=None, *,
                       cyclic=False,
                       w_target=None,
                       B_size=20, p=3, iters=2000, list_limit=5000, verbose=True):
    """
    Given P from the LWPM solver, build a generator matrix G from P
    and try to find a low-weight codeword of weight <= w_target using ISD.

    If n is None, default to n = degree(PQ) + 1, which matches  n = t + d + 1.
    If w_target is None, default to weight(PQ) (try to match or beat it).
    """
    R = P.parent()
    if n is None:
        n = PQ.degree() + 1

    # Build generator matrix
    G = generator_matrix_from_polynomial(P, n, cyclic=cyclic)

    if verbose:
        print(f"Constructed G of shape {G.nrows()} x {G.ncols()} (cyclic={cyclic}).")
        print(f"Target weight ≤ {w_target} (PQ achieved this or higher).")

    ok, c, u, wt, stats = stern_codeword_search(
        G, w_target, B_size=B_size, p=p, list_limit=list_limit, iters=iters, verbose=verbose
    )
    return ok, c, u, wt, stats, G


from sage.all import GF, PolynomialRing

F2 = GF(2)
R = PolynomialRing(F2, 'x')
x = R.gen()

def list_to_poly(L):
    return sum((c % 2) * x**i for i, c in enumerate(L))


if __name__ == '__main__':
    # read from file
    in_dir = r""
    pq_in_dir = r""

    # input files
    p_path = os.path.join(in_dir, "coeffs_p.txt")
    pq_path = os.path.join(pq_in_dir, "pq_norm.txt")

    # create directory to a specific path
    out_dir = r""
    os.makedirs(out_dir, exist_ok=True)


    t = 200  # Degree of P
    d = 60  # Degree of Q
    w = 10  # Max Hamming weight
    n=60

    P_all = read_P_file(p_path)
    PQ_all=read_weight_file(pq_path)

    for i, (P, PQ) in enumerate(zip(P_all, PQ_all)):
        with \
            open(os.path.join(out_dir, "pq.txt"), "a+", encoding="utf-8") as f_PQ, \
            open(os.path.join(out_dir, "pq_norm.txt"), "a+", encoding="utf-8") as f_norm, \
            open(os.path.join(out_dir, "q.txt"), "a+", encoding="utf-8") as f_Q, \
            open(os.path.join(out_dir, "wall_time.txt"), "a+", encoding="utf-8") as f_wall, \
            open(os.path.join(out_dir, "cpu_time.txt"), "a+", encoding="utf-8") as f_cpu, \
            open(os.path.join(out_dir, "peak_mem.txt"), "a+", encoding="utf-8") as f_mem:

                print(f"Instance {i + 1}/{len(P_all)}")
                seed(i)

                P  = list_to_poly(P)
                
                print("P=", P)
                print("Pq=",PQ)

                G = generator_matrix_from_polynomial(P, n, cyclic=False)

                try:
                    result = run_with_timeout(
                        120,   # 2 minute
                        run_isd_after_lwpm,
                        P=P,
                        n=n,
                        cyclic=False,
                        w_target=PQ,
                        B_size=20,
                        p=2,
                        iters=200,
                        list_limit=5000,
                        verbose=True
                    )

                    if result["status"] == "timeout":
                        print("Timeout: no solution found in 2 minutes.")

                        f_wall.write("TIMEOUT\n")
                        f_cpu.write("TIMEOUT\n")
                        f_mem.write("TIMEOUT\n")

                        f_PQ.write(str([0]) + "\n")
                        f_norm.write("TIMEOUT\n")
                        f_Q.write(str([0]) + "\n")

                        continue

                    elif result["status"] == "error":
                        print(result["message"])
                        continue

                    measure = result["data"]

                    ok, c, u, wt, stats, G = measure["result"]
                    
                    R.<x> = PolynomialRing(GF(2))

                    c_poly = R(list(c))
                    u_poly = R(list(u))
                    
                    print(c_poly)
                    print(u_poly)

                    print(f"Wall time   : {measure['wall_time_s']:.3f} s")
                    print(f"CPU time    : {measure['cpu_time_s']:.3f} s")
                    print(f"Peak memory : {measure['peak_mem_MiB']:.1f} MiB")

                    f_wall.write(f"{measure['wall_time_s']}\n")
                    f_cpu.write(f"{measure['cpu_time_s']}\n")
                    f_mem.write(f"{measure['peak_mem_MiB']}\n")

                    if ok:
                        print("Found codeword with weight <=", hamming_weight_vec(c))
                        print("c=", c)
                        print("u=", u)

                        f_PQ.write(str(c) + "\n")
                        f_norm.write(str(wt) + "\n")
                        f_Q.write(str(u) + "\n")
                    else:
                        print("No improvement within the given budget.")
                        f_PQ.write(str([0]) + "\n")
                        f_norm.write("0\n")
                        f_Q.write(str([0]) + "\n")

                except Exception as e:
                    print(e)
                    print("Solver error, skipping instance", i)
                    continue 
